# Similarity

In [1]:
import torch
import torch.nn.functional as F
from pathlib import Path
from transformers import AutoModel, AutoTokenizer

/Users/thangtran/Workplace/master_s_degree/information_retrieval/backend/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_NAME = "vinai/phobert-base"

CACHE_DIR = "../../data/models/vinai-phobert"

phoBert = AutoModel.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)

phoBert.eval()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 51354.61it/s]
[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.decoder.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RobertaModel(
  (embeddings): RobertaEmbeddings(
    (word_embeddings): Embedding(64001, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (position_embeddings): Embedding(258, 768, padding_idx=1)
  )
  (encoder): RobertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x RobertaLayer(
        (attention): RobertaAttention(
          (self): RobertaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): RobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=Tru

In [3]:
EMBEDDING_DIR = Path("../../data/embeddings")

V1_PATH = EMBEDDING_DIR / "v1_sentence_level_retrieval_embeddings.pt"
V2_PATH = EMBEDDING_DIR / "v2_document_level_retrieval_embedding.pt"

v1_data = torch.load(V1_PATH, weights_only=False)
v2_data = torch.load(V2_PATH, weights_only=False)

sentence_embeddings = v1_data["embeddings"]
sentences = v1_data["sentences"]
document_embedding = v2_data["embedding"]

print("V1 sentence embeddings:", sentence_embeddings.shape)
print("V2 document embedding: ", document_embedding.shape)

V1 sentence embeddings: torch.Size([34, 768])
V2 document embedding:  torch.Size([768])


In [4]:
def mean_pooling(
    token_embeddings: torch.Tensor,
    pooling_mask: torch.Tensor
) -> torch.Tensor:
    mask = pooling_mask.unsqueeze(-1).float()

    sum_embeddings = torch.sum(
        token_embeddings * mask,
        dim=1
    )

    token_count = torch.sum(
        mask,
        dim=1
    ).clamp(min=1e-9)

    return sum_embeddings / token_count

In [5]:
def encode_query(query: str) -> torch.Tensor:
    encoded = tokenizer(
        query,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )

    with torch.no_grad():
        output = phoBert(**encoded)

    token_embeddings = output.last_hidden_state

    special_tokens_mask = torch.tensor([
        tokenizer.get_special_tokens_mask(
            ids,
            already_has_special_tokens=True
        )
        for ids in encoded["input_ids"].tolist()
    ])

    content_mask = (
        encoded["attention_mask"]
        * (1 - special_tokens_mask)
    )

    query_embedding = mean_pooling(
        token_embeddings,
        content_mask
    )

    return query_embedding.squeeze(0)


In [6]:
query = "Giá vé máy_bay đi Singapore giảm mạnh ."

query_embedding = encode_query(query)

print("Query:")
print(query)

print("\nQuery embedding shape:")
print(query_embedding.shape)


Query:
Giá vé máy_bay đi Singapore giảm mạnh .

Query embedding shape:
torch.Size([768])


# V1

Sử dụng Cosine Similarity với từng sentence

In [7]:
v1_scores = F.cosine_similarity(
    sentence_embeddings,
    query_embedding.unsqueeze(0),
    dim=1
)

print("Scores shape:")
print(v1_scores.shape)

print("\nScores:")
print(v1_scores)


Scores shape:
torch.Size([34])

Scores:
tensor([0.7426, 0.7050, 0.7514, 0.7053, 0.6871, 0.6084, 0.6668, 0.7782, 0.7411,
        0.7099, 0.7818, 0.6803, 0.7813, 0.7004, 0.6443, 0.7440, 0.7251, 0.6581,
        0.6589, 0.6323, 0.6246, 0.7392, 0.7130, 0.7622, 0.6313, 0.6447, 0.6630,
        0.7058, 0.6098, 0.6244, 0.6947, 0.6319, 0.6710, 0.7169])


Ranking các sentence

In [8]:
ranking = torch.argsort(
    v1_scores,
    descending=True
)

for rank, sentence_index in enumerate(ranking.tolist(), start=1):
    score = v1_scores[sentence_index].item()

    print(
        f"{rank}. score={score:.4f} | "
        f"sentence_id={sentence_index}"
    )
    print(f"   {sentences[sentence_index]}\n")


1. score=0.7818 | sentence_id=10
   Từ Hà_Nội đi Singapore và Thái_Lan , giá vé của các hãng trong nước dao_động 2,6-3 triệu đồng một_chiều , đã gồm thuế , phí .

2. score=0.7813 | sentence_id=12
   Với đường_bay TP HCM - Jakarta , giá cũng giảm nhưng mặt_bằng vẫn cao hơn Singapore và Thái_Lan .

3. score=0.7782 | sentence_id=7
   Khảo_sát các đường_bay từ TP HCM đi Singapore và Thái_Lan cho thấy mức giá khuyến_mại 19.000-90.000 đồng , chưa gồm thuế , phí chiếm đa_số các chặng bay trong tháng 8 và 9 .

4. score=0.7622 | sentence_id=23
   Ông Hồng_Thanh , chủ một đại_lý vé máy_bay tại TP HCM , cho biết nguồn cung tăng và cạnh_tranh giữa các hãng là nguyên_nhân quan_trọng khiến giá vé quốc_tế hạ nhiệt .

5. score=0.7514 | sentence_id=2
   Theo chị , các khoản phí tại sân_bay Singapore cao hơn chiều bay từ Việt_Nam nên dù cùng giá vé niêm_yết , số tiền thực trả vẫn chênh_lệch đáng_kể .

6. score=0.7440 | sentence_id=15
   Các đường_bay từ Hà_Nội và TP HCM tới châu_Âu , Đông_Bắc_Á cũng giả

Document Score bằng MAX

Baseline:

```text
Document Score = max(sentence similarity)
```


In [9]:
v1_document_score = v1_scores.max().item()
best_sentence_index = v1_scores.argmax().item()

print(f"V1 document score: {v1_document_score:.4f}")
print(f"Best sentence ID: {best_sentence_index}")
print(sentences[best_sentence_index])


V1 document score: 0.7818
Best sentence ID: 10
Từ Hà_Nội đi Singapore và Thái_Lan , giá vé của các hãng trong nước dao_động 2,6-3 triệu đồng một_chiều , đã gồm thuế , phí .


# V2

Cosine Similarity với document vector

In [10]:
v2_document_score = F.cosine_similarity(
    query_embedding.unsqueeze(0),
    document_embedding.unsqueeze(0),
    dim=1
).item()

print(f"V2 document score: {v2_document_score:.4f}")


V2 document score: 0.8082


# Compare versions

In [11]:
print("=== RESULT ===")
print(f"Query:   {query}")
print()

print(
    f"V1 - Sentence-level (MAX): "
    f"{v1_document_score:.4f}"
)

print(
    f"V2 - Document-level:       "
    f"{v2_document_score:.4f}"
)

=== RESULT ===
Query:   Giá vé máy_bay đi Singapore giảm mạnh .

V1 - Sentence-level (MAX): 0.7818
V2 - Document-level:       0.8082
